# Notebook 3: Autoaprendizaje

## Pre-requisitos

### Instalar paquetes

Si la práctica requiere algún paquete de Python, habrá que incluir una celda en la que se instalen. Si usamos un paquete que se ha utilizado en prácticas anteriores, podríamos dar por supuesto que está instalado pero no cuesta nada satisfacer todas las dependencias en la propia práctica para reducir las dependencias entre ellas.

In [1]:
# instalación TensorFlow
# !pip3 install tensorflow
import tensorflow as tf

# Hacemos los imports que sean necesarios
import numpy as np

# Autoaprendizaje sobre Fashion-MNIST

Lo primero que tenemos que hacer es cargar el dataset.

In [2]:
labeled_data = 0.05 # Vamos a usar el etiquetado de sólo el 5% de los datos
np.random.seed(42)

(x_train, y_train), (x_test, y_test), = tf.keras.datasets.fashion_mnist.load_data()

indexes = np.arange(len(x_train))
np.random.shuffle(indexes)
ntrain_data = int(labeled_data*len(x_train))
unlabeled_train = x_train[indexes[ntrain_data:]]
x_train = x_train[indexes[:ntrain_data]]
y_train = y_train[indexes[:ntrain_data]]

In [3]:
# TODO: Haz el preprocesado que necesites aquí (si lo necesitas)

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Pasamos de dimensión 3 (n_samples, 28, 28) a dim 2 (n_samples, 748)
x_train = x_train.reshape(len(x_train), -1)
unlabeled_train = unlabeled_train.reshape(len(unlabeled_train), -1)
x_test = x_test.reshape(len(x_test), -1)

# convertimos de uint8 a float32
x_train = x_train.astype("float32")
unlabeled_train = unlabeled_train.astype("float32")
x_test = x_test.astype("float32")

# normalizamos y aplicamos PCA
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
unlabeled_train_scaled = scaler.transform(unlabeled_train)
x_test_scaled = scaler.transform(x_test)

pca = PCA(n_components=50, random_state=42)
x_train_pca = pca.fit_transform(x_train_scaled)
unlabeled_train_pca = pca.transform(unlabeled_train_scaled)
x_test_pca = pca.transform(x_test_scaled)

## Función de autoaprendizaje

Vamos a crear nuestra propia función de autoaprendizaje. Para ello, vamos a utilizar el siguiente pseudocódigo.



**self_training** *(model, x_train, y_train, unlabeled_data, thresh, train_epochs)*

1. $train\_data, train\_label \leftarrow x\_train, y\_train$
1. **Desde** $n = 1 .. train\_epochs$ **hacer**
	1. Entrena el *model* usando las variables *train\_data* y *train\_label* 
	2. $y\_pred \leftarrow model(unlabeled\_data)$
  
  3. $y\_class, y\_value \leftarrow $ Clase ganadora en *y_pred* con su valor

  4. $train\_data, train\_label \leftarrow x\_train, y\_train$
  
  5. **Para cada elemento** (x_u, y_c, y_v) **de la tupla** (unlabeled_data, y_class, y_value) 
	  1. **Si** $y\_v > thresh$ **entonces**
		
        1. Añadimos $x\_u$ e $y\_c$ a train\_data y train\_label, respectivamente.

4. Devolvemos el modelo


<font color='red'>NOTA:</font> para entrenar (y predecir) vamos a utilizar los modelos de Sklearn. Familiarízate con las funciones [fit](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC.fit), [predict](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC.predict), [predict_proba](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC.predict_proba) y [score](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC.score).

In [10]:
# TODO: implementa el algoritmo self_training tal y como viene en el pseudocódigo. Las variables extra son para epara visualización de resultados

def self_training(model_func, x_train, y_train, unlabeled_data, x_test, y_test, thresh=0.5, train_epochs=3):
    train_data = x_train.copy()
    train_label = y_train.copy()

    for i in range(train_epochs):
        print("EPOCH: ", i)
        model = model_func()
        model.fit(train_data, train_label)

        y_pred = model.predict_proba(unlabeled_data)

        y_class = np.argmax(y_pred, axis=1)
        y_value = np.max(y_pred, axis=1)

        train_data = x_train.copy()
        train_label = y_train.copy()

        for x_u, y_c, y_v in zip(unlabeled_data, y_class, y_value):
            if y_v > thresh:
                train_data = np.append(train_data, x_u.reshape(1, -1), axis=0) # usamos reshape y axis para añadir fila con append en array
                train_label = np.append(train_label, y_c)

    return model

### Entrenamos nuestro clasificador

Usa lo hecho anteriormente para entrenar tu clasificador de una manera semi-supervisada. Utiliza para ello el [SVM](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) de sklearn (vigila el parámetro probability).

In [8]:
# Define la función para llamar al SVM
from sklearn.svm import SVC
model_func = lambda: SVC(kernel='linear', probability=True, max_iter=200)

In [12]:
# TODO: Entrena tu clasificador

model = self_training(
    model_func,
    x_train,
    y_train,
    unlabeled_train,
    x_test,
    y_test,
    thresh=0.9,
    train_epochs=3
)

acc = model.score(x_test, y_test)
print("Accuracy:", acc)

EPOCH:  0


c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


EPOCH:  1


c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


EPOCH:  2


c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Accuracy: 0.7795


## Mejorando el código

Tal como hemos visto en clase de teoría, este código puede ser mejorado de varias maneras:

  1. **Asignar más peso a las variables etiquetadas**.
  1. **Asignar un peso en función de la certeza de la predicción en las variables sin etiquetar**.

### TRABAJO: Modifica la función self_training para tener en cuenta todos los puntos mencionados anteriormente

In [14]:
# TODO: reescribe la función self_training para incorporar las mejoras mencionadas anteriormente

def self_training_v2(model_func, x_train, y_train, unlabeled_data, x_test, y_test, thresh=0.5, train_epochs=3):

    train_data = x_train.copy()
    train_label = y_train.copy()
    train_weights = np.ones(len(train_label))  # todos los etiquetados con peso 1

    for i in range(train_epochs):
        print("EPOCH: ", i)
        model = model_func()
        model.fit(train_data, train_label, sample_weight=train_weights)
        y_pred = model.predict_proba(unlabeled_data)

        y_class = np.argmax(y_pred, axis=1)
        y_value = np.max(y_pred, axis=1)

        for i in range(len(unlabeled_data)):
            prob = y_value[i]
            if prob > thresh:
                train_data = np.append(train_data, unlabeled_data[i].reshape(1, -1), axis=0)
                train_label = np.append(train_label, y_class[i])
                train_weights = np.append(train_weights, prob)

    return model

In [16]:
# TODO: Entrena tu clasificador

model = self_training_v2(
    model_func,
    x_train,
    y_train,
    unlabeled_train,
    x_test,
    y_test,
    thresh=0.9,
    train_epochs=3
)

acc = model.score(x_test, y_test)
print("Accuracy:", acc)

EPOCH:  0


c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


EPOCH:  1


c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


EPOCH:  2


c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Accuracy: 0.7689


### Creamos nuestro clasificador en Tensorflow

Ya deberías de ser capaz de realizar este trabajo con muy poca ayuda. El diseño del clasificador es libre (capas densas, convolucionales, ...), puedes crearlo como quieras. Lo único que tenemos que tener en cuenta es que tenemos que encapsular nuestro modelo en una clase cuyas funciones tengan el mismo nombre (y los mismos parámetros) que las existentes en sklearn.

In [ ]:
# TODO: crea tu propio clasificador

class MiClasificador:

    def __init__(self):
        # TODO : define el modelo
        self.model = None
        # TODO: crea el optimizador
        None
        # TODO: compila el modelo
        self.model.compile(
            loss=None,
            optimizer=optimizer,
            metrics=['accuracy']
        )
    
    def fit(self, X, y, sample_weight=None):
        # TODO: entrena el modelo. Escoge el tamaño de batch y el número de epochs que quieras
        pass

    def predict(self, X):
        # TODO: devuelve la clase ganadora
        pass
    
    def predict_proba(self, X):
        # TODO: devuelve las probabilidades de cada clase
        pass
    
    def score(self, X, y):
        # TODO: devuelve el accuracy del clasificador
        pass

    def __del__(self):
        del self.model
        tf.keras.backend.clear_session() # Necesario para liberar la memoria en GPU

### Entrenando el modelo

Crea una función que nos permita crear el modelo en cada iteración.

In [ ]:
# TODO: Entrena el modelo
None